In [3]:
import geopandas as gpd
from pyogrio import read_info

path = "../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2020.shp"
# adjust the path if your notebook is somewhere else; use an absolute path if unsure

info = read_info(path)
print(info["features"], "features")
print(info["fields"])
print(info["dtypes"])

head = gpd.read_file(path, rows=5)
print(head.drop(columns="geometry"))
print(head.crs)

941872 features
['fire_ID' 'lat' 'lon' 'size' 'perimeter' 'start_date' 'start_DOY'
 'end_date' 'end_DOY' 'duration' 'fire_line' 'spread' 'speed' 'direction'
 'direc_frac' 'MODIS_tile' 'landcover' 'landc_frac' 'GFED_regio']
['int32' 'float64' 'float64' 'float64' 'float64' 'object' 'int32' 'object'
 'int32' 'int32' 'float64' 'float64' 'float64' 'object' 'float64' 'object'
 'object' 'float64' 'int32']
   fire_ID      lat      lon  size  perimeter  start_date  start_DOY  \
0    76988 -20.0104 -61.4981  1.50       5.56  2020-05-04        125   
1    76989 -20.0020 -59.4640  0.43       2.78  2020-09-23        267   
2    76995 -20.0020 -57.7790  3.64       9.26  2020-09-25        269   
3    76996 -20.0020 -57.6238  0.21       1.85  2020-06-26        178   
4    77012 -20.0020 -56.8922  0.43       2.78  2020-06-08        160   

     end_date  end_DOY  duration  fire_line  spread  speed direction  \
0  2020-05-08      129         5       0.65    0.30   0.67     north   
1  2020-09-23      26

In [5]:
import geopandas as gpd, pandas as pd

base = "../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_{}.shp"
frames = []
for yr in (2019, 2020, 2021, 2022):
    df = gpd.read_file(base.format(yr), ignore_geometry=True)
    df["start_date"] = pd.to_datetime(df["start_date"])
    df["file_year"] = yr
    print(yr, len(df), df["start_date"].min().date(), df["start_date"].max().date())
    frames.append(df)

fires = pd.concat(frames, ignore_index=True)
print("duplicate fire_IDs across files:", fires["fire_ID"].duplicated().sum())
print(fires["landcover"].value_counts())
print(fires["size"].describe())

2019 883922 2018-07-01 2020-05-31
2020 941872 2019-07-01 2021-05-31
2021 811489 2020-07-01 2022-05-31
2022 799297 2021-07-01 2023-05-31
duplicate fire_IDs across files: 2533584
landcover
Savannas                       1403956
Grasslands                      942749
Croplands                       376933
Woody savannas                  332643
Evergreen Broadleaf forest      141256
Deciduous Broadleaf forest      111465
Mixed forest                     67234
Open shrublands                  30203
Unclassified                      9836
Water                             7743
Evergreen Needleleaf forest       4590
Closed shrublands                 3576
Urban and built-up                3248
Deciduous Needleleaf forest       1136
Fill Value                          12
Name: count, dtype: int64
count    3.436580e+06
mean     4.017334e+00
std      2.736845e+01
min      2.100000e-01
25%      2.100000e-01
50%      8.600000e-01
75%      2.360000e+00
max      1.796905e+04
Name: size, dtype: float64

In [6]:
sel = fires[
    (fires["start_date"] >= "2020-01-01") & (fires["start_date"] < "2022-01-01")
    & (fires["size"] >= 5)          # about 500 ha, if size is in km2
].copy()
print(len(sel))
print(sel.groupby("landcover").size().sort_values(ascending=False))

239428
landcover
Savannas                       102524
Grasslands                      77903
Croplands                       18545
Woody savannas                  16846
Deciduous Broadleaf forest       9034
Mixed forest                     6014
Evergreen Broadleaf forest       3570
Open shrublands                  3567
Closed shrublands                 476
Unclassified                      447
Evergreen Needleleaf forest       386
Water                              51
Urban and built-up                 37
Deciduous Needleleaf forest        28
dtype: int64


In [7]:
dups = fires[fires.duplicated("fire_ID", keep=False)]
g = dups.groupby("fire_ID").agg(
    n=("file_year", "size"),
    n_lat=("lat", "nunique"),
    n_start=("start_date", "nunique"),
    n_size=("size", "nunique"),
)
print(g["n"].value_counts())
print((g[["n_lat", "n_start", "n_size"]] > 1).mean())   # share of IDs whose fields differ

n
4    759867
2     72024
5     39384
3     12189
6         9
Name: count, dtype: int64
n_lat      1.000000
n_start    0.999973
n_size     0.978144
dtype: float64


In [8]:
before = len(fires)
fires_u = fires.drop_duplicates(
    subset=["lat", "lon", "start_date", "end_date", "size"]).copy()
print("rows before:", before, " after:", len(fires_u), " removed:", before - len(fires_u))

# create your own unique ID, since fire_ID can't serve as one
fires_u["uid"] = fires_u["file_year"].astype(str) + "_" + fires_u["fire_ID"].astype(str)
print("uid unique:", fires_u["uid"].is_unique)

rows before: 3436580  after: 3397170  removed: 39410
uid unique: True


In [9]:
forest = ["Evergreen Broadleaf forest", "Deciduous Broadleaf forest", "Mixed forest",
          "Evergreen Needleleaf forest", "Deciduous Needleleaf forest"]

sel = fires_u[
    (fires_u["start_date"] >= "2020-01-01") & (fires_u["start_date"] < "2022-01-01")
    & (fires_u["size"] >= 5)
    & (fires_u["landcover"].isin(forest))
    & (fires_u["landc_frac"] >= 0.5)
].copy()

print(len(sel))
print(sel.groupby("landcover").size())
print(sel.groupby("GFED_regio").size())
print(sel["size"].describe())

16240
landcover
Deciduous Broadleaf forest     7736
Deciduous Needleleaf forest      16
Evergreen Broadleaf forest     3048
Evergreen Needleleaf forest     349
Mixed forest                   5091
dtype: int64
GFED_regio
1      124
2      238
3      287
4       59
5     1986
6       15
7       10
8     2167
9     7161
10     181
11      79
12    3781
13      17
14     135
dtype: int64
count    16240.000000
mean        19.328871
std         43.430675
min          5.140000
25%          6.860000
50%         10.290000
75%         18.860000
max       2977.800000
Name: size, dtype: float64


In [10]:
strata = (sel.groupby(["GFED_regio", "landcover"]).size()
             .rename("n_fires").reset_index())
print(strata.sort_values(["GFED_regio", "n_fires"], ascending=[True, False]).to_string(index=False))

top = (sel.sort_values("size", ascending=False)
          .groupby(["GFED_regio", "landcover"]).head(3)
          .sort_values(["GFED_regio", "landcover", "size"], ascending=[True, True, False]))
print(top[["uid", "GFED_regio", "landcover", "lat", "lon",
           "start_date", "end_date", "size"]].to_string(index=False))

 GFED_regio                   landcover  n_fires
          1 Evergreen Needleleaf forest      108
          1                Mixed forest       15
          1  Deciduous Broadleaf forest        1
          2 Evergreen Needleleaf forest      145
          2  Deciduous Broadleaf forest       51
          2  Evergreen Broadleaf forest       30
          2                Mixed forest       12
          3  Evergreen Broadleaf forest      141
          3  Deciduous Broadleaf forest      127
          3                Mixed forest       19
          4  Evergreen Broadleaf forest       51
          4  Deciduous Broadleaf forest        8
          5  Evergreen Broadleaf forest     1569
          5  Deciduous Broadleaf forest      406
          5                Mixed forest        9
          5 Evergreen Needleleaf forest        2
          6 Evergreen Needleleaf forest        9
          6  Deciduous Broadleaf forest        4
          6                Mixed forest        2
          7 Evergree

In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)
sel.to_parquet("../data/processed/forest_fires_2020_2021_attrs.parquet")

In [12]:
import geopandas as gpd, os

path = "../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2020.shp"
pilot = gpd.read_file(path, engine="pyogrio", where="fire_ID = 112")

print(pilot.drop(columns="geometry").T)   # check start_date 2020-08-14, size 1206.04
print(pilot.total_bounds)                 # minx, miny, maxx, maxy
os.makedirs("../data/processed/pilot", exist_ok=True)
pilot.to_file("../data/processed/pilot/fire_2020_112.gpkg", driver="GPKG")

                                      0
fire_ID                             112
lat                             39.6938
lon                            -122.782
size                            1206.04
perimeter                        308.36
start_date                   2020-08-14
start_DOY                           227
end_date                     2020-09-27
end_DOY                             271
duration                             45
fire_line                         39.34
spread                             26.8
speed                              4.42
direction                     northwest
direc_frac                         0.37
MODIS_tile                       h08v05
landcover   Evergreen Needleleaf forest
landc_frac                          0.5
GFED_regio                            2
[-123.09298503   39.5125     -122.55808696   39.87916666]


In [15]:
import rasterio
from rasterio.windows import from_bounds

url = "https://glad.geog.umd.edu/Potapov/Forest_height_2019/Forest_height_2019_NAM.tif"
west, south, east, north = -123.093, 39.5125, -122.558, 39.8792

with rasterio.open(url) as src:
    print(src.crs, src.res)
    win = from_bounds(west, south, east, north, src.transform)
    h = src.read(1, window=win)
print(h.shape)

EPSG:4326 (0.00025, 0.00025)
(1467, 2140)


In [16]:
import numpy as np
valid = h[h <= 60]          # 101 water, 102 snow/ice, 103 no data
print("share valid:", valid.size / h.size)
print("height percentiles (5, 25, 50, 75, 95):", np.percentile(valid, [5, 25, 50, 75, 95]))

share valid: 0.999959546152425
height percentiles (5, 25, 50, 75, 95): [ 0.  7. 15. 24. 34.]
